### CEO Supervisor Agent

Creates a Multi-Agent Supervisor (MAS) for the CEO Dashboard that orchestrates:
- **Revenue Analytics** (Revenue Genie) — real-time revenue, orders, location performance
- **Operations Intelligence** (Operations Genie) — complaints, refunds, kitchen health
- **Inspection Reports** (Inspection KA) — food safety compliance documents
- **Legal Complaints** (Legal KA) — litigation, employment disputes, liability claims
- **Regulatory Documents** (Regulatory KA) — permits, certifications, compliance status
- **Audit Reports** (Audit KA) — financial and operational audit findings
- **Consultancy Reports** (Consultancy KA) — strategic recommendations

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG              = dbutils.widgets.get("CATALOG")
SUPERVISOR_ENDPOINT  = dbutils.widgets.get("CEO_SUPERVISOR_ENDPOINT_NAME")
LEGAL_ENDPOINT       = dbutils.widgets.get("CEO_LEGAL_KA_ENDPOINT")
REG_ENDPOINT         = dbutils.widgets.get("CEO_REGULATORY_KA_ENDPOINT")
AUDIT_ENDPOINT       = dbutils.widgets.get("CEO_AUDIT_KA_ENDPOINT")
CONSULT_ENDPOINT     = dbutils.widgets.get("CEO_CONSULTANCY_KA_ENDPOINT")
INSPECTION_ENDPOINT  = dbutils.widgets.get("INSPECTION_KNOWLEDGE_ENDPOINT_NAME")

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
API_BASE = "/api/2.0/multi-agent-supervisors"

##### Retrieve Genie space IDs from uc_state

In [ ]:
revenue_title = f"CEO Revenue & Orders Intelligence ({CATALOG})"
ops_title = f"CEO Operations Intelligence ({CATALOG})"

revenue_genie_id = None
ops_genie_id = None

df = spark.sql(f"""
    SELECT resource_data FROM {CATALOG}._internal_state.resources
    WHERE resource_type = 'genie_spaces'
    ORDER BY created_at DESC
""")

for row in df.collect():
    info = json.loads(row.resource_data)
    if info.get("title") == revenue_title and not revenue_genie_id:
        revenue_genie_id = info.get("space_id")
    if info.get("title") == ops_title and not ops_genie_id:
        ops_genie_id = info.get("space_id")

if not revenue_genie_id:
    raise RuntimeError(f"Revenue Genie space not found in uc_state. Run ceo_genie_spaces stage first.")
if not ops_genie_id:
    raise RuntimeError(f"Operations Genie space not found in uc_state. Run ceo_genie_spaces stage first.")

print(f"Revenue Genie: {revenue_genie_id}")
print(f"Operations Genie: {ops_genie_id}")

##### Resolve KA endpoint IDs from the tiles API

In [ ]:
def _list_kas():
    """List all KAs via the KA API (handles pagination)."""
    kas, params = [], {}
    try:
        while True:
            resp = w.api_client.do("GET", "/api/2.0/knowledge-assistants", query=params)
            kas.extend(resp.get("knowledge_assistants", []))
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  KA list error: {e}")
    return kas


def _list_mas():
    """List all MAS tiles via the MAS API (handles pagination)."""
    mas, params = [], {}
    try:
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            mas.extend(resp.get("multi_agent_supervisors", []))
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  MAS list error: {e}")
    return mas


def get_tile_by_name(name):
    """Fallback: search /api/2.0/tiles by name."""
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == name:
                    return tile
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception:
        pass
    return None


def resolve_endpoint(ka_name):
    """Return the serving endpoint name for a KA by its name."""
    # Primary: KA API
    for item in _list_kas():
        ka = item.get("knowledge_assistant", item)
        if ka.get("name") == ka_name:
            ep = ka.get("endpoint_name") or item.get("endpoint_name")
            if ep:
                return ep
    # Fallback: generic tiles API
    tile = get_tile_by_name(ka_name)
    if tile:
        ep = tile.get("serving_endpoint_name") or tile.get("endpoint_name")
        if ep:
            return ep
    raise RuntimeError(f"No KA found for '{ka_name}'. Ensure KA stage ran first.")


def find_mas_by_endpoint(endpoint_name):
    """Find an existing MAS tile_id by endpoint_name."""
    for item in _list_mas():
        mas = item.get("multi_agent_supervisor", item)
        if mas.get("endpoint_name") == endpoint_name:
            return item.get("tile_id") or mas.get("tile_id") or mas.get("id")
    return None


def find_mas_by_name(name):
    """Find an existing MAS tile_id by name."""
    for item in _list_mas():
        mas = item.get("multi_agent_supervisor", item)
        if mas.get("name") == name:
            return item.get("tile_id") or mas.get("tile_id") or mas.get("id")
    # Fallback: tiles API
    tile = get_tile_by_name(name)
    if tile:
        return tile.get("tile_id")
    return None


legal_ep_id    = resolve_endpoint(f"{CATALOG}-ceo-legal")
reg_ep_id      = resolve_endpoint(f"{CATALOG}-ceo-regulatory")
audit_ep_id    = resolve_endpoint(f"{CATALOG}-ceo-audits")
consult_ep_id  = resolve_endpoint(f"{CATALOG}-ceo-consultancy")
inspect_ep_id  = resolve_endpoint(f"{CATALOG}-inspection-knowledge")

print(f"Legal KA endpoint:      {legal_ep_id}")
print(f"Regulatory KA endpoint: {reg_ep_id}")
print(f"Audit KA endpoint:      {audit_ep_id}")
print(f"Consultancy KA endpoint:{consult_ep_id}")
print(f"Inspection KA endpoint: {inspect_ep_id}")


##### Create the CEO Supervisor Agent

In [ ]:
import time

AGENT_NAME = f"{CATALOG}-ceo-supervisor"

EXAMPLES = [
    {
        "question": "Give me the executive briefing on the state of the business",
        "guideline": "Should summarize revenue performance, top operational risks, any critical legal or compliance issues, and one strategic recommendation. Keep it to 5 bullet points maximum."
    },
    {
        "question": "Which location is most at risk right now?",
        "guideline": "Should consider complaint rate, inspection score, legal exposure, and audit findings for each location. Rank and explain the top risk with specific numbers."
    },
    {
        "question": "Which location has the highest order cancellation rate right now?",
        "guideline": "Must name exactly one specific location with the highest rate. Must include the numeric cancellation rate as a percentage. Must not say it cannot access the data. Routes to revenue-analytics."
    },
    {
        "question": "How does revenue compare across our locations this week?",
        "guideline": "Must provide revenue figures or ranking for multiple locations. Must reference the current week. Must not return a generic description without actual numbers. Routes to revenue-analytics."
    },
    {
        "question": "Which brand is generating the most revenue right now?",
        "guideline": "Must name a specific brand (not a location). Must include a revenue figure or ranking. Must distinguish between brand-level and location-level performance. Routes to revenue-analytics."
    },
    {
        "question": "Which location needs the most operational attention right now?",
        "guideline": "Must name one specific location with the highest operational risk. Must justify with at least two operational metrics (e.g. complaint rate, cancel rate, food safety score). Must reference actual data. Routes to operations-intelligence."
    },
    {
        "question": "What happened during the Chicago food safety inspection? Were there critical violations?",
        "guideline": "Must reference a specific inspection report by date or ID. Must state the inspection score and/or grade. Must explicitly state whether critical violations were found and cite at least one specific violation code if they exist. Routes to inspection-reports."
    },
    {
        "question": "Do we have any active high-risk legal cases? What is the total financial exposure?",
        "guideline": "Must confirm whether active cases exist. Must cite at least one specific case number (format CK-XX-XXXX). Must include a risk classification (HIGH/MEDIUM/LOW) for at least one case. Must state a financial exposure amount. Must remind CEO to involve legal counsel. Routes to legal-complaints."
    },
    {
        "question": "Give me all the legal cases for Chicago",
        "guideline": "Must list the legal complaint filings (CK-04-XXXX case numbers) from the legal-complaints agent — not food safety violations or customer complaint counts. Must include case type, status, and amount at stake for each case found. Routes to legal-complaints only."
    },
    {
        "question": "Which location has the most active legal cases?",
        "guideline": "Must name a specific location. Must provide a count of active legal filings (CK-XX-XXXX format) for at least the top location. Must distinguish active cases from settled or dismissed ones. Must not conflate food safety violations or customer complaints with legal cases. Routes to legal-complaints."
    },
    {
        "question": "Are there any permits or regulatory certificates expiring in the next 60 days?",
        "guideline": "Must list specific document IDs or permit names. Must include expiry dates for each flagged document. Must specify which location each document applies to. If nothing is expiring, must say so explicitly. Routes to regulatory-compliance."
    },
    {
        "question": "What were the most significant audit findings this quarter?",
        "guideline": "Must cite the auditing firm and audit period. Must classify findings by severity (Critical/Significant/Minor/Informational). Must state remediation status for at least one finding. Must not conflate audit findings with food safety inspection findings. Routes to audit-findings."
    },
    {
        "question": "What do our consultants recommend as the top AI investments for the next 90 days?",
        "guideline": "Must reference a specific consulting report or firm. Must include at least one concrete recommendation. Must include a financial metric (ROI estimate, cost, or projected saving). Must frame the answer in the 90-day horizon. Routes to consultancy-strategy."
    },
    {
        "question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation",
        "guideline": "Must explicitly address all four domains: revenue, operations, legal, and strategy. Must include at least one concrete number or data point per domain. Must not refuse or hedge on any domain. Response must be structured with clear sections, not a single unbroken paragraph. Routes to multiple agents."
    },
]

SUPERVISOR_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Executive AI assistant for the CEO of Casper's Kitchens. Synthesizes intelligence "
        "across revenue analytics, operations, food safety, legal exposure, regulatory compliance, "
        "audit findings, and strategic consulting recommendations across all 8 ghost kitchen "
        "locations (US: San Francisco, Silicon Valley, Bellevue, Chicago; EMEA: London, Munich, Amsterdam, Vianen) and 16 restaurant brands."
    ),
    "endpoint_name": SUPERVISOR_ENDPOINT,
    "agents": [
        {
            "agent_type": "genie",
            "genie_space": {"id": revenue_genie_id},
            "name": "revenue-analytics",
            "description": (
                "ONLY call this agent for questions explicitly about: revenue ($), sales totals, order counts, average order value, brand sales rankings, location revenue comparisons, or financial performance metrics. Keywords that should trigger this agent: revenue, sales, earned, orders, top performing, best/worst location by revenue, avg order, financial. Do NOT call for food safety, inspections, legal, audits, or strategic questions."
            ),
        },
        {
            "agent_type": "genie",
            "genie_space": {"id": ops_genie_id},
            "name": "operations-intelligence",
            "description": (
                "ONLY call this agent for questions about: order throughput, kitchen operations, delivery performance, food safety inspection scores, food safety violation counts, location health metrics, or operational risk. "
                "Keywords: operations, kitchen, delivery, inspection score, food safety grade, food safety violations, throughput, busiest hours, peak demand, operational issues, cancel rate, complaint rate. "
                "Do NOT call for revenue/financial totals, legal cases, lawsuits, legal exposure, legal issues, audit reports, or strategy. "
                "IMPORTANT: 'complaints' in this agent means customer operational complaints (cancellations, refunds) — NOT legal filings or lawsuits. For anything involving courts, case numbers, or litigation, use legal-complaints instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": inspect_ep_id},
            "name": "inspection-reports",
            "description": (
                "ONLY call for questions about specific food safety inspection documents, detailed inspection findings, corrective action details from inspectors, or historical inspection report text. Keywords: inspection report, inspector findings, corrective action, food safety document. For inspection SCORES/GRADES use operations-intelligence instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": legal_ep_id},
            "name": "legal-complaints",
            "description": (
                "ONLY call for questions about lawsuits, legal cases, customer complaint filings, litigation status, legal liability, legal issues, or specific complaint case numbers. "
                "Keywords: lawsuit, legal complaint, litigation, legal case, legal issues, legal exposure, filed complaint, liability, customer liability claims, vendor contract disputes, legal financial exposure, legal risk, court case. "
                "When asked to list or enumerate cases for a specific location (e.g. 'all legal issues in Chicago'), search for '[location name] legal case' or '[location name] complaint' and summarise all cases found, including case number, type, status, and amount. "
                "Always surface case numbers (format CK-XX-XXXX), risk levels (HIGH/MEDIUM/LOW), and amounts at stake."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": reg_ep_id},
            "name": "regulatory-compliance",
            "description": (
                "ONLY call for questions about permits, licenses, certifications, regulatory compliance documents, or specific regulatory requirements. Keywords: permit, license, certification, regulatory, compliance document, regulation."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": audit_ep_id},
            "name": "audit-findings",
            "description": (
                "ONLY call for questions about financial or operational audit reports, audit findings, audit recommendations, or specific audit case references. Keywords: audit, audit report, audit finding, auditor recommendation."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": consult_ep_id},
            "name": "consultancy-strategy",
            "description": (
                "ONLY call for questions about strategic recommendations, consultant advice, improvement strategies, or specific consultancy report references. Keywords: consultant, consultancy report, strategic recommendation, improvement strategy, advisor."
            ),
        },
    ],
    "instructions": (
        "You are an elite executive AI assistant serving the CEO of Casper's Kitchens, a ghost kitchen "
        "network operating 16 restaurant brands across 8 locations (US: San Francisco, Silicon Valley, "
        "Bellevue, Chicago; EMEA: London, Munich, Amsterdam, Vianen). "
        "Synthesize intelligence from multiple sources to deliver concise, executive-ready insights. "
        "Always specify which location(s) you are referencing. "
        "Lead with the most important finding, then provide supporting detail. "
        "IMPORTANT DATA SOURCE BOUNDARIES: "
        "(1) 'Legal issues', 'lawsuits', 'legal cases', 'legal exposure' always refer to formal legal complaint filings (CK-XX-XXXX case numbers) retrieved from the legal-complaints agent — NOT food safety violations or customer operational complaints. "
        "(2) 'Food safety violations' are inspection findings retrieved from operations-intelligence or inspection-reports — they are compliance issues, not lawsuits. "
        "(3) 'Customer complaints' (complaint rate, cancel rate) are operational metrics from operations-intelligence — they are not legal filings. "
        "Never conflate food safety violations or operational complaint counts with legal case filings. "
        "When discussing legal or audit matters, remind the CEO to involve counsel or auditors for decisions. "
        "When the conversation reveals performance issues, flag whether the CEO should consider engaging "
        "consultants, streamlining current engagements, or accelerating AI investments. "
        "Be direct, data-driven, and strategic — as a top McKinsey partner would be."
    ),
}


# Look up existing MAS directly via the MAS API — most reliable source of truth
existing_id = find_mas_by_endpoint(SUPERVISOR_ENDPOINT) or find_mas_by_name(AGENT_NAME)
agent_id = None

if existing_id:
    try:
        w.api_client.do("PUT", f"{API_BASE}/{existing_id}", body=SUPERVISOR_BODY)
        agent_id = existing_id
        print(f"\u267b\ufe0f  Updated existing MAS: {agent_id}")
    except Exception as e:
        print(f"  PUT failed for {existing_id}: {e} \u2014 will create new")

if not agent_id:
    try:
        w.api_client.do("POST", API_BASE, body=SUPERVISOR_BODY)
        print(f"\u2705 Created CEO Supervisor")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"\u267b\ufe0f Supervisor already exists (race condition), proceeding")
        else:
            raise

tile = get_tile_by_name(AGENT_NAME)
if tile:
    agent_id = tile["tile_id"]
    print(f"Resolved tile_id: {agent_id}")
else:
    agent_id = AGENT_NAME
    print(f"\u26a0\ufe0f Using name as fallback: {agent_id}")

# MAS endpoints are created with the pattern mas-{first-8-chars-of-tile-id}-endpoint
# (same convention as KA endpoints: ka-{tile_id[:8]}-endpoint)
if isinstance(agent_id, str) and len(agent_id) >= 8 and "-" in agent_id:
    actual_endpoint_name = f"mas-{agent_id[:8]}-endpoint"
else:
    actual_endpoint_name = SUPERVISOR_ENDPOINT
print(f"MAS serving endpoint name: {actual_endpoint_name}")

add(CATALOG, "multi_agent_supervisors", {
    "endpoint_name": actual_endpoint_name,
    "tile_id": agent_id,
    "name": AGENT_NAME,
})

# Poll for the serving endpoint to be READY, then add examples via the tiles API.
# Examples are NOT accepted in the create/update body — they require a separate call
# to POST /api/2.0/tiles/{tile_id}/examples once the endpoint is online.
if agent_id and agent_id != AGENT_NAME and actual_endpoint_name:
    print(f"\nPolling MAS endpoint readiness ({actual_endpoint_name})...")
    ep_ready = False
    for attempt in range(1, 61):
        try:
            resp = w.api_client.do("GET", f"/api/2.0/serving-endpoints/{actual_endpoint_name}")
            state = resp.get("state", {})
            ready = str(state.get("ready", "")).upper()
            if ready == "READY":
                print(f"  ✅ Endpoint READY")
                ep_ready = True
                break
            print(f"  [{attempt}/60] state={ready} — waiting 20s…")
        except Exception as e:
            print(f"  [{attempt}/60] poll error: {e} — waiting 20s…")
        time.sleep(20)

    if ep_ready:
        try:
            w.api_client.do(
                "POST",
                f"/api/2.0/tiles/{agent_id}/examples",
                body={"examples": EXAMPLES},
            )
            print(f"✅ Examples added ({len(EXAMPLES)} questions)")
        except Exception as e:
            print(f"⚠️  Could not add examples: {e}")
    else:
        print(f"⚠️  Endpoint not READY after 20 min — skipping examples (re-run to retry)")

print(f"\n✅ CEO Supervisor stage complete")
print(f"   Endpoint: {actual_endpoint_name}")
print(f"   Agents: revenue-analytics, operations-intelligence, inspection-reports, legal-complaints, regulatory-compliance, audit-findings, consultancy-strategy")

# Write the resolved tile_id into app.yaml so the dashboard can build the
# correct /ml/bricks/sa/<tile_id> deep link without any runtime API lookups.
import os, re as _re

app_yaml_path = "../apps/ceo-dashboard/app.yaml"
try:
    with open(app_yaml_path) as _f:
        _yaml = _f.read()

    # Also resolve the MLflow experiment ID from the tile
    _mlflow_exp_id = ""
    try:
        _t = get_tile_by_name(AGENT_NAME)
        if _t:
            _mlflow_exp_id = str(_t.get("mlflow_experiment_id") or "")
    except Exception:
        pass

    _yaml = _re.sub(
        r'(- name: CEO_SUPERVISOR_TILE_ID\s*\n\s*value:) .*',
        rf'\1 \'{agent_id}\'',
        _yaml,
    )
    _yaml = _re.sub(
        r'(- name: CEO_SUPERVISOR_ENDPOINT\s*\n\s*value:) .*',
        rf'\1 \'{actual_endpoint_name}\'',
        _yaml,
    )
    if _mlflow_exp_id:
        _yaml = _re.sub(
            r'(- name: CEO_SUPERVISOR_MLFLOW_EXPERIMENT_ID\s*\n\s*value:) .*',
            rf'\1 \'{_mlflow_exp_id}\'',
            _yaml,
        )
    with open(app_yaml_path, "w") as _f:
        _f.write(_yaml)
    print(f"✅ app.yaml updated: tile_id={agent_id}, endpoint={actual_endpoint_name}, mlflow_experiment_id={_mlflow_exp_id}")
except Exception as _e:
    print(f"⚠️  Could not update app.yaml: {_e} — set CEO_SUPERVISOR_TILE_ID manually")

# Redeploy the CEO Dashboard app so it picks up the updated CEO_SUPERVISOR_TILE_ID
# and CEO_SUPERVISOR_ENDPOINT from app.yaml.
_app_name = "ceo-dashboard"
_app_src  = os.path.abspath("../apps/ceo-dashboard")
try:
    from databricks.sdk.service.apps import AppDeployment as _AppDeployment
    _dep = w.apps.deploy(
        app_name=_app_name,
        app_deployment=_AppDeployment(source_code_path=_app_src),
    )
    _dep.result()
    print(f"✅ CEO Dashboard redeployed (tile_id={agent_id})")
except Exception as _deploy_err:
    print(f"⚠️  Could not redeploy app automatically: {_deploy_err}")
    print(f"   Run ceo_app.ipynb or redeploy '{_app_name}' from the Databricks Apps UI.")